<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/Gradio_Blocks_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Today we will build a UI App which uses our fine tuned model via Gradio Blocks Framework

In [2]:
# Install dependencies
%pip install -q transformers torch gradio "pandas<3.0.0"

print(f'✅ Installed Dependencies Successfully!')

✅ Installed Dependencies Successfully!


In [45]:
# Import packages
from transformers import pipeline
import gradio as gr
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

print(f'✅ Imports successful!')


✅ Imports successful!


In [46]:
classifier = pipeline("text-classification", model="abhishes/novapay-sentiment")

print(f'✅ Classifier loaded successfully')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Classifier loaded successfully


In [5]:
# verify versions

print(f'Gradio Version: {gr.__version__}')

Gradio Version: 6.20.0


In [47]:
# write classification function

def classify(text, threshold, history):
  if not text.strip():
    return {"no input": 1.0}, gr.update(visible=False, value=""), history, history

  result = classifier(text)[0]
  retVal = "Positive" if result['label'] == "LABEL_1" else "Negative"
  label_dict= {retVal: result['score']}

  # update history
  history.append({
    "Message": text[:60] + ("..." if len(text) > 60 else ""),
    "Label": retVal,
    "Score": f"{result['score']:0.4f}"
  })


  if result['score'] < threshold:
    warning = f"⚠️ Low confidence ({result['score']:.3f} < {threshold}) — Manual review recommended"
    return label_dict, gr.update(visible=True, value=warning), pd.DataFrame(history), history
  else:
    return label_dict, gr.update(visible=False, value=""), pd.DataFrame(history), history


In [42]:
with gr.Blocks("NovaPay Sentiment Analyzer v 1.0") as demo:
  gr.Markdown("# NovaPay Sentiment Analyzer v1.0")

  #history
  history = gr.State([])

  with gr.Row():
    with gr.Column():
      input_text = gr.Textbox(label="Customer Message: ", placeholder="Type a NovaPay customer message...", lines=5)
      threshold_slider = gr.Slider(minimum=0.5, maximum=0.99, value=0.8, step=0.01, label="Confidence Threshold")
      submit_button = gr.Button("🔎 Analyze Message", variant="primary")
    with gr.Column():
      output_label = gr.Label(label="Sentiment Result")
      warning_box = gr.Textbox(label="⚠️ Alert", visible=False, interactive=False)
  with gr.Row():
    gr.Markdown("### 📊 Classfication History")
  with gr.Row():
    history_df = gr.Dataframe(headers=["Message", "Label", "Score"], label="Session History")
    submit_button.click(fn=classify, inputs=[input_text, threshold_slider, history], outputs=[output_label, warning_box, history_df, history])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0fb7ed31143935dfe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [44]:
demo.close()